In [1]:
import torch
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.vfae.adapter import KatabaticVFAE  # Changed to VFAE Adapter
from utils import discretize_preprocess

# --- Configuration ---
DATASET = "adult"
raw_path = f"raw_data/{DATASET}.csv"
discretized_path = f"discretized_data/{DATASET}.csv"
output_dir = f"sample_data/{DATASET}"
synthetic_dir = f"synthetic/{DATASET}/vfae"  # Updated directory
real_test_dir = f"sample_data/{DATASET}"

# Device Config
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# User Input (with default for safety)
protected_col = input(f"Protected Attribute (S) [default 'sex']: ").strip() or "sex"
target_col = input(f"Target Attribute (Y) [default 'class']: ").strip() or "class"

# --- Model Configuration ---
model_config = {
    # VFAE Hyperparameters
    "epochs": 50,
    "batch_size": 64,
    "z_dim": 50,         # Latent dimension size
    
    # Fairness Config
    # REQUIRED by VFAE to split input data into X, S, Y components
    # REQUIRED by FairnessEvaluation to calculate metrics
    "fairness_config": {
        "S": protected_col,       # Sensitive Column Name
        "Y": target_col,          # Target Column Name
        "S_under": "0",           # Value representing unprivileged group (e.g., '0' for Female)
        "Y_desire": "1"           # Value representing positive outcome (optional context)
    }
}

# --- 1. Preprocess ---
print("Discretizing data...")
discretize_preprocess(
    file_path=raw_path,
    output_path=discretized_path,
    bins=10,
    strategy='uniform'
)

# --- 2. Run Pipeline ---
# Instantiate pipeline with the VFAE Adapter class
pipeline = TrainTestSplitPipeline(model=KatabaticVFAE)

print(f"Starting pipeline for VFAE on {DATASET}")

pipeline.run(
    input_csv=discretized_path,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    fairness_eval=False,
    **model_config
)

/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Discretizing data...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Starting pipeline for VFAE on adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Loading VFAE training data from: sample_data/adult
Initializing VFAE (X:13, S:1, Y:1)...


Training VFAE:   0%|          | 0/50 [00:00<?, ?it/s]/home/adity/github/katabatic-mentorship-repo/katabatic/models/vfae/utils.py:16: UserWarning: Using a target size (torch.Size([64, 1])) that is different to the input size (torch.Size([64, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  sup_loss = F.mse_loss(outputs['y_decoded'], y_target, reduction='sum')
Training VFAE: 100%|██████████| 50/50 [01:15<00:00,  1.52s/it, loss=3715.4544]


Generating synthetic data to: synthetic/adult/vfae
Saved artifacts: x_synth.csv ((26048, 14)), y_synth.csv ((26048,))

Results saved to: Results/adult/vfae_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6720
F1 Score: 0.6956
AUC: 0.7363

MLP:
Accuracy: 0.6926
F1 Score: 0.7138
AUC: 0.7625

RF:
Accuracy: 0.6340
F1 Score: 0.6588
AUC: 0.7964

XGBoost:
Accuracy: 0.6043
F1 Score: 0.6287
AUC: 0.7719


'Train test split pipeline executed successfully.'